In [1]:
import json
import os
import yaml
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Tuple, Union

import draccus
import torch
import torch.distributed as dist
from torch.utils.data import DataLoader, Dataset, DistributedSampler
from tqdm import tqdm
from transformers.modeling_outputs import CausalLMOutputWithPast
from transformers.models.auto import AutoConfig
from PIL import Image
from torch.distributed.fsdp import FullyShardedDataParallel as FSDP

In [2]:
from prismatic.conf import DatasetConfig, DatasetRegistry, ModelConfig, ModelRegistry
from prismatic.models import get_llm_backbone_and_tokenizer, get_vision_backbone_and_transform, get_vlm
from prismatic.overwatch import initialize_overwatch
from prismatic.preprocessing import get_dataset_and_collator
from prismatic.training import Metrics, get_train_strategy
from prismatic.util import set_global_seed

# Load a checkpoint

In [3]:
checkpoint_path = "../runs/dino+siglip-llama-42/checkpoints/latest-checkpoint.pt"

In [3]:
checkpoint_path = "../runs/align-reproduction-llava-v15+7b-llava-v15-42/checkpoints/latest-checkpoint.pt"

In [4]:
checkpoint = torch.load(checkpoint_path, map_location="cuda" if torch.cuda.is_available() else "cpu")

In [5]:
checkpoint['model']['llm_backbone'].keys()

odict_keys(['llm.base_model.model.model.embed_tokens.weight', 'llm.base_model.model.model.layers.0.self_attn.q_proj.weight', 'llm.base_model.model.model.layers.0.self_attn.k_proj.weight', 'llm.base_model.model.model.layers.0.self_attn.v_proj.weight', 'llm.base_model.model.model.layers.0.self_attn.o_proj.base_layer.weight', 'llm.base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'llm.base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'llm.base_model.model.model.layers.0.mlp.gate_proj.weight', 'llm.base_model.model.model.layers.0.mlp.up_proj.weight', 'llm.base_model.model.model.layers.0.mlp.down_proj.base_layer.weight', 'llm.base_model.model.model.layers.0.mlp.down_proj.lora_A.default.weight', 'llm.base_model.model.model.layers.0.mlp.down_proj.lora_B.default.weight', 'llm.base_model.model.model.layers.0.input_layernorm.weight', 'llm.base_model.model.model.layers.0.post_attention_layernorm.weight', 'llm.base_model.model.model.layers.1.self_attn.

In [5]:
def check_checkpoint(checkpoint_path: Path):
    print(f"[DEBUG] From {checkpoint_path}")
    print(f"[DEBUG] Checkpoint global_step = {checkpoint.get('global_step', 'MISSING')}")
    print(f"[DEBUG] Checkpoint epoch = {checkpoint.get('epoch', 'MISSING')}")
    print(f"[DEBUG] Checkpoint samples_seen = {checkpoint.get('samples_seen', 'MISSING')}")
    print(f"[DEBUG] Everything else...\n{checkpoint.keys()}")

In [6]:
check_checkpoint(checkpoint_path=checkpoint_path)

[DEBUG] From ../runs/dino+siglip-llama-42/checkpoints/latest-checkpoint.pt
[DEBUG] Checkpoint global_step = 2500
[DEBUG] Checkpoint epoch = 0
[DEBUG] Checkpoint samples_seen = 160000
[DEBUG] Everything else...
dict_keys(['model', 'optimizer', 'lr_scheduler', 'epoch', 'global_step', 'samples_seen', 'rng_state', 'train_loss'])


In [6]:
check_checkpoint(checkpoint_path=checkpoint_path)

[DEBUG] From ../runs/align-reproduction-llava-v15+7b-llava-v15-42/checkpoints/latest-checkpoint.pt
[DEBUG] Checkpoint global_step = 2000
[DEBUG] Checkpoint epoch = 0
[DEBUG] Checkpoint samples_seen = 256000
[DEBUG] Everything else...
dict_keys(['model', 'lr_scheduler', 'epoch', 'global_step', 'samples_seen', 'rng_state', 'train_loss'])


In [7]:
from prismatic import load

In [ ]:
load("../runs/align-reproduction-llava-v15+7b-llava-v15-42")

09/14 [23:24:35] INFO     | >> [*] Loading from local path                                               ]8;id=518025;file:///share/data/speech/txu/vlm_semantics/prismatic-vlms/prismatic/models/load.py\load.py]8;;\:]8;id=934562;file:///share/data/speech/txu/vlm_semantics/prismatic-vlms/prismatic/models/load.py#53\53]8;;\
                          `../runs/align-reproduction-llava-v15+7b-llava-v15-42`                                   

                 INFO     | >> [*] Found Config =>> Loading & Freezing reproduction-llava-v15+7b with:   ]8;id=958387;file:///share/data/speech/txu/vlm_semantics/prismatic-vlms/prismatic/models/load.py\load.py]8;;\:]8;id=899525;file:///share/data/speech/txu/vlm_semantics/prismatic-vlms/prismatic/models/load.py#75\75]8;;\
                                       Vision Backbone =>> clip-vit-l-336px                                        
                                       LLM Backbone    =>> vicuna-v15-7b                                           
                                       Arch Specifier  =>> gelu-mlp                                                
                                       Checkpoint Path =>>                                                         
                          `../runs/align-reproduction-llava-v15+7b-llava-v15-42/checkpoints/latest-check           
                          point.pt`                                                                                

                 INFO     | >> [*] Loading Vision Backbone clip-vit-l-336px                              ]8;id=741836;file:///share/data/speech/txu/vlm_semantics/prismatic-vlms/prismatic/models/load.py\load.py]8;;\:]8;id=56851;file:///share/data/speech/txu/vlm_semantics/prismatic-vlms/prismatic/models/load.py#84\84]8;;\

09/14 [23:24:41] INFO     | >> Loading pretrained weights from Hugging Face hub                     ]8;id=23131;file:///share/data/speech/txu/vlm_semantics/venv/lib/python3.11/site-packages/timm/models/_builder.py\_builder.py]8;;\:]8;id=616428;file:///share/data/speech/txu/vlm_semantics/venv/lib/python3.11/site-packages/timm/models/_builder.py#186\186]8;;\
                          (('timm/vit_large_patch14_clip_336.openai',                                              
                          'open_clip_pytorch_model.bin'))                                                          

09/14 [23:24:42] INFO     | >>  Safe alternative available for 'open_clip_pytorch_model.bin' (as        ]8;id=280220;file:///share/data/speech/txu/vlm_semantics/venv/lib/python3.11/site-packages/timm/models/_hub.py\_hub.py]8;;\:]8;id=146160;file:///share/data/speech/txu/vlm_semantics/venv/lib/python3.11/site-packages/timm/models/_hub.py#180\180]8;;\
                          'open_clip_model.safetensors'). Loading weights using safetensors.                       

09/14 [23:25:14] INFO     | >> [*] Loading Pretrained LLM vicuna-v15-7b via HF Transformers              ]8;id=764372;file:///share/data/speech/txu/vlm_semantics/prismatic-vlms/prismatic/models/load.py\load.py]8;;\:]8;id=692247;file:///share/data/speech/txu/vlm_semantics/prismatic-vlms/prismatic/models/load.py#91\91]8;;\

                 INFO     | >>     |=> Building empty llama2 LLM from `lmsys/vicuna-7b-v1.5`        ]8;id=546424;file:///share/data/speech/txu/vlm_semantics/prismatic-vlms/prismatic/models/backbones/llm/base_llm.py\base_llm.py]8;;\:]8;id=476861;file:///share/data/speech/txu/vlm_semantics/prismatic-vlms/prismatic/models/backbones/llm/base_llm.py#136\136]8;;\

09/14 [23:29:36] INFO     | >>     |=> Loading llama2 (Fast) Tokenizer via the AutoTokenizer API    ]8;id=270229;file:///share/data/speech/txu/vlm_semantics/prismatic-vlms/prismatic/models/backbones/llm/base_llm.py\base_llm.py]8;;\:]8;id=922298;file:///share/data/speech/txu/vlm_semantics/prismatic-vlms/prismatic/models/backbones/llm/base_llm.py#155\155]8;;\

09/14 [23:30:10] INFO     | >> [*] Loading VLM reproduction-llava-v15+7b from Checkpoint; Freezing      ]8;id=835073;file:///share/data/speech/txu/vlm_semantics/prismatic-vlms/prismatic/models/load.py\load.py]8;;\:]8;id=969356;file:///share/data/speech/txu/vlm_semantics/prismatic-vlms/prismatic/models/load.py#100\100]8;;\
                          Weights 🥶                                                                               

# Debug the Optimizer

In [13]:
len(checkpoint['optimizer']['state'])

1

In [18]:
checkpoint['optimizer']['state']

{0: {'step': tensor(2500., device='cuda:0'),
  'exp_avg': tensor([-1.5963e-07, -1.9706e-06, -8.6609e-07,  ..., -6.5393e-07,
          -3.1674e-07,  3.5618e-07], device='cuda:0'),
  'exp_avg_sq': tensor([9.5170e-11, 1.9516e-10, 1.2621e-10,  ..., 2.6558e-10, 9.5376e-10,
          4.0572e-11], device='cuda:0')}}

In [17]:
checkpoint['optimizer']['state'][0].keys()

dict_keys(['step', 'exp_avg', 'exp_avg_sq'])

In [15]:
checkpoint['optimizer']['state'][0]['exp_avg'].shape

torch.Size([17846400])